# 🌿 WhatAPlant — Entraînement de Modèle Machine Learning

**Objectif :** Entraîner un modèle capable de prédire si une plante est **comestible** à partir des données collectées via l'application WhatAPlant.

**Problème ML :** Classification binaire / multi-label
- Entrées : `confidence` PlantNet, `family`, `is_toxic`, `is_medicinal`
- Sorties : `is_edible` (Oui / Non / Partiellement)

---
**Stack :** Python · Pandas · Scikit-learn · Matplotlib · Seaborn

## 1. 📦 Imports & Configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Scikit-learn
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.pipeline import Pipeline

# Style graphiques
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
sns.set_theme(style='whitegrid', palette='Set2')

print('✅ Imports OK')

## 2. 🗄️ Chargement des données

Deux sources :
1. **Données réelles** depuis MySQL (scans WhatAPlant)
2. **Données synthétiques** enrichies pour un jeu d'entraînement robuste

In [ ]:
import pymysql
import os
from dotenv import load_dotenv

# Charger les variables d'environnement
load_dotenv('../backend/.env')

def load_real_data():
    """Charge les scans réels depuis MySQL."""
    try:
        conn = pymysql.connect(
            host=os.getenv('MYSQL_HOST', 'localhost'),
            port=int(os.getenv('MYSQL_PORT', 3306)),
            user=os.getenv('MYSQL_USER', 'root'),
            password=os.getenv('MYSQL_PASSWORD', ''),
            database=os.getenv('MYSQL_DB', 'whataPlant'),
            charset='utf8mb4',
            cursorclass=pymysql.cursors.DictCursor
        )
        with conn.cursor() as c:
            c.execute("""
                SELECT scientific_name, family, confidence,
                       is_edible, is_medicinal, is_toxic, is_invasive
                FROM scans
                WHERE is_edible IS NOT NULL AND is_edible != ''
            """)
            rows = c.fetchall()
        conn.close()
        df = pd.DataFrame(rows)
        df['source'] = 'réelle'
        print(f'✅ {len(df)} scans réels chargés depuis MySQL')
        return df
    except Exception as e:
        print(f'⚠️  MySQL indisponible : {e}')
        print('   → Utilisation des données synthétiques uniquement')
        return pd.DataFrame()

df_real = load_real_data()
df_real.head()

In [ ]:
# ── Génération des données synthétiques ──────────────────────────────────
# Basées sur des connaissances botaniques réelles (Afrique de l'Ouest)

np.random.seed(42)

plants_data = [
    # (scientific_name, family, is_edible, is_medicinal, is_toxic, is_invasive, base_confidence)
    ('Mangifera indica',       'Anacardiaceae',  'Oui',          'Oui', 'Non', 'Non', 88),
    ('Moringa oleifera',       'Moringaceae',    'Oui',          'Oui', 'Non', 'Non', 91),
    ('Carica papaya',          'Caricaceae',     'Oui',          'Oui', 'Non', 'Non', 85),
    ('Hibiscus sabdariffa',    'Malvaceae',      'Oui',          'Oui', 'Non', 'Non', 79),
    ('Cassava esculenta',      'Euphorbiaceae',  'Partiellement', 'Non', 'Oui', 'Non', 72),
    ('Datura stramonium',      'Solanaceae',     'Non',          'Non', 'Oui', 'Non', 65),
    ('Ricinus communis',       'Euphorbiaceae',  'Non',          'Non', 'Oui', 'Non', 70),
    ('Vernonia amygdalina',    'Asteraceae',     'Oui',          'Oui', 'Non', 'Non', 83),
    ('Ocimum gratissimum',     'Lamiaceae',      'Oui',          'Oui', 'Non', 'Non', 87),
    ('Chromolaena odorata',    'Asteraceae',     'Non',          'Non', 'Non', 'Oui', 60),
    ('Lantana camara',         'Verbenaceae',    'Non',          'Non', 'Oui', 'Oui', 68),
    ('Colocasia esculenta',    'Araceae',        'Oui',          'Non', 'Non', 'Non', 80),
    ('Solanum nigrum',         'Solanaceae',     'Partiellement', 'Oui', 'Oui', 'Non', 55),
    ('Calotropis procera',     'Apocynaceae',    'Non',          'Oui', 'Oui', 'Non', 74),
    ('Parkia biglobosa',       'Fabaceae',       'Oui',          'Oui', 'Non', 'Non', 92),
    ('Vitellaria paradoxa',    'Sapotaceae',     'Oui',          'Oui', 'Non', 'Non', 89),
    ('Adansonia digitata',     'Malvaceae',      'Oui',          'Oui', 'Non', 'Non', 95),
    ('Azadirachta indica',     'Meliaceae',      'Non',          'Oui', 'Oui', 'Non', 86),
    ('Xanthosoma sagittifolium','Araceae',       'Oui',          'Non', 'Non', 'Non', 77),
    ('Eichhornia crassipes',   'Pontederiaceae', 'Non',          'Non', 'Non', 'Oui', 58),
    ('Amaranthus hybridus',    'Amaranthaceae',  'Oui',          'Oui', 'Non', 'Non', 81),
    ('Tithonia diversifolia',  'Asteraceae',     'Non',          'Oui', 'Non', 'Non', 63),
    ('Jatropha curcas',        'Euphorbiaceae',  'Non',          'Non', 'Oui', 'Non', 76),
    ('Ipomoea batatas',        'Convolvulaceae', 'Oui',          'Non', 'Non', 'Non', 90),
    ('Abrus precatorius',      'Fabaceae',       'Non',          'Non', 'Oui', 'Non', 66),
    ('Ceiba pentandra',        'Malvaceae',      'Oui',          'Oui', 'Non', 'Non', 82),
    ('Khaya senegalensis',     'Meliaceae',      'Non',          'Oui', 'Non', 'Non', 71),
    ('Ficus exasperata',       'Moraceae',       'Partiellement', 'Oui', 'Non', 'Non', 69),
    ('Trichilia emetica',      'Meliaceae',      'Non',          'Oui', 'Non', 'Non', 64),
    ('Sorghum bicolor',        'Poaceae',        'Oui',          'Non', 'Non', 'Non', 93),
]

rows_synth = []
for plant in plants_data:
    name, family, edible, medicinal, toxic, invasive, base_conf = plant
    # Générer 15 échantillons par plante avec variation de confiance
    for _ in range(15):
        conf = np.clip(base_conf + np.random.normal(0, 5), 20, 99)
        rows_synth.append({
            'scientific_name': name,
            'family': family,
            'confidence': round(conf, 1),
            'is_edible': edible,
            'is_medicinal': medicinal,
            'is_toxic': toxic,
            'is_invasive': invasive,
            'source': 'synthétique'
        })

df_synth = pd.DataFrame(rows_synth)
print(f'✅ {len(df_synth)} données synthétiques générées ({len(plants_data)} espèces × 15 échantillons)')

In [ ]:
# ── Fusion des deux sources ───────────────────────────────────────────────
if not df_real.empty:
    df = pd.concat([df_real, df_synth], ignore_index=True)
    print(f'Dataset combiné : {len(df_real)} réels + {len(df_synth)} synthétiques = {len(df)} total')
else:
    df = df_synth.copy()
    print(f'Dataset synthétique uniquement : {len(df)} entrées')

print(f'\nDistribution is_edible :')
print(df['is_edible'].value_counts())

## 3. 🔍 Exploration et analyse des données (EDA)

In [ ]:
print('=== Aperçu du dataset ===')
print(f'Forme : {df.shape}')
print(f'Colonnes : {list(df.columns)}')
print(f'\nValeurs manquantes :')
print(df.isnull().sum())
df.head(10)

In [ ]:
print('=== Statistiques descriptives ===')
df[['confidence']].describe().round(2)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('📊 Analyse exploratoire — Dataset WhatAPlant', fontsize=16, fontweight='bold')

# 1. Distribution is_edible
counts = df['is_edible'].value_counts()
colors = ['#2ecc71', '#e74c3c', '#f39c12']
axes[0,0].pie(counts.values, labels=counts.index, autopct='%1.1f%%', colors=colors, startangle=90)
axes[0,0].set_title('Distribution : Comestibilité')

# 2. Distribution de la confiance PlantNet
axes[0,1].hist(df['confidence'], bins=20, color='#3498db', edgecolor='white', alpha=0.85)
axes[0,1].set_xlabel('Score de confiance PlantNet (%)')
axes[0,1].set_ylabel('Nombre d\'observations')
axes[0,1].set_title('Distribution du score de confiance')
axes[0,1].axvline(df['confidence'].mean(), color='red', linestyle='--', label=f'Moy = {df["confidence"].mean():.1f}%')
axes[0,1].legend()

# 3. Confiance selon comestibilité
df.boxplot(column='confidence', by='is_edible', ax=axes[1,0],
           boxprops=dict(color='#2c3e50'), medianprops=dict(color='red', linewidth=2))
axes[1,0].set_title('Confiance selon la comestibilité')
axes[1,0].set_xlabel('Comestible')
axes[1,0].set_ylabel('Confiance (%)')
plt.sca(axes[1,0])
plt.title('Confiance selon la comestibilité')

# 4. Top familles botaniques
top_families = df['family'].value_counts().head(8)
axes[1,1].barh(top_families.index, top_families.values, color='#9b59b6', alpha=0.85)
axes[1,1].set_xlabel('Nombre d\'observations')
axes[1,1].set_title('Top 8 familles botaniques')
for i, v in enumerate(top_families.values):
    axes[1,1].text(v + 0.5, i, str(v), va='center')

plt.tight_layout()
plt.savefig('eda_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Graphique EDA sauvegardé')

In [ ]:
# Matrice de corrélation entre les propriétés de la plante
bool_map = {'Oui': 1, 'Non': 0, 'Partiellement': 0.5}
df_corr = df[['confidence', 'is_edible', 'is_medicinal', 'is_toxic', 'is_invasive']].copy()
for col in ['is_edible', 'is_medicinal', 'is_toxic', 'is_invasive']:
    df_corr[col] = df_corr[col].map(bool_map).fillna(0)

plt.figure(figsize=(8, 6))
mask = np.triu(np.ones_like(df_corr.corr(), dtype=bool))
sns.heatmap(df_corr.corr(), annot=True, fmt='.2f', cmap='RdYlGn',
            mask=mask, vmin=-1, vmax=1, linewidths=0.5,
            xticklabels=['Confiance', 'Comestible', 'Médicinal', 'Toxique', 'Invasif'],
            yticklabels=['Confiance', 'Comestible', 'Médicinal', 'Toxique', 'Invasif'])
plt.title('🔗 Corrélation entre les propriétés des plantes', fontweight='bold')
plt.tight_layout()
plt.savefig('correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. 🛠️ Préparation des données d'entraînement

In [ ]:
# ── Nettoyage ─────────────────────────────────────────────────────────────
print('Avant nettoyage :', df.shape)

# Supprimer les lignes avec valeurs critiques manquantes
df_clean = df.dropna(subset=['is_edible', 'confidence', 'family']).copy()

# Normaliser les labels
df_clean['is_edible']    = df_clean['is_edible'].str.strip().str.capitalize()
df_clean['is_medicinal'] = df_clean['is_medicinal'].fillna('Non').str.strip().str.capitalize()
df_clean['is_toxic']     = df_clean['is_toxic'].fillna('Non').str.strip().str.capitalize()
df_clean['is_invasive']  = df_clean['is_invasive'].fillna('Non').str.strip().str.capitalize()

# Normaliser les valeurs
valid_labels = {'Oui', 'Non', 'Partiellement'}
df_clean = df_clean[df_clean['is_edible'].isin(valid_labels)]

print('Après nettoyage :', df_clean.shape)
print('\nDistribution finale :')
print(df_clean['is_edible'].value_counts())

In [ ]:
# ── Feature Engineering ──────────────────────────────────────────────────

# Encoder les variables catégorielles
le_family   = LabelEncoder()
le_medicinal = LabelEncoder()
le_toxic    = LabelEncoder()
le_invasive = LabelEncoder()

df_clean['family_enc']    = le_family.fit_transform(df_clean['family'])
df_clean['medicinal_enc'] = le_medicinal.fit_transform(df_clean['is_medicinal'])
df_clean['toxic_enc']     = le_toxic.fit_transform(df_clean['is_toxic'])
df_clean['invasive_enc']  = le_invasive.fit_transform(df_clean['is_invasive'])

# Feature : confiance normalisée par rapport à la famille
family_mean = df_clean.groupby('family')['confidence'].transform('mean')
df_clean['confidence_vs_family'] = df_clean['confidence'] - family_mean

# Features finales
FEATURES = [
    'confidence',
    'family_enc',
    'medicinal_enc',
    'toxic_enc',
    'invasive_enc',
    'confidence_vs_family'
]

# Encoder la cible
le_target = LabelEncoder()
TARGET = 'is_edible'

X = df_clean[FEATURES]
y = le_target.fit_transform(df_clean[TARGET])

print(f'Features utilisées : {FEATURES}')
print(f'Classes cibles     : {list(le_target.classes_)}')
print(f'X shape : {X.shape}')
print(f'y shape : {y.shape}')

In [ ]:
# ── Séparation Train / Test ───────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Jeu d\'entraînement : {X_train.shape[0]} échantillons ({X_train.shape[0]/len(X)*100:.0f}%)')
print(f'Jeu de test        : {X_test.shape[0]} échantillons ({X_test.shape[0]/len(X)*100:.0f}%)')
print(f'\nDistribution dans le train :')
for i, label in enumerate(le_target.classes_):
    n = (y_train == i).sum()
    print(f'  {label} : {n} ({n/len(y_train)*100:.1f}%)')

## 5. 🤖 Entraînement et comparaison des modèles

In [ ]:
# ── Définition des modèles à comparer ─────────────────────────────────────
models = {
    'Logistic Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(max_iter=500, random_state=42))
    ]),
    'Random Forest': Pipeline([
        ('model', RandomForestClassifier(n_estimators=100, random_state=42))
    ]),
    'Gradient Boosting': Pipeline([
        ('model', GradientBoostingClassifier(n_estimators=100, random_state=42))
    ]),
    'SVM': Pipeline([
        ('scaler', StandardScaler()),
        ('model', SVC(kernel='rbf', probability=True, random_state=42))
    ]),
}

# ── Entraînement et évaluation de chaque modèle ───────────────────────────
results = {}

for name, pipeline in models.items():
    print(f'\n⏳ Entraînement : {name}...')
    
    # Entraîner
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    
    # Validation croisée (5-fold)
    cv_scores = cross_val_score(pipeline, X, y, cv=5, scoring='f1_weighted')
    
    # Métriques
    results[name] = {
        'pipeline': pipeline,
        'y_pred': y_pred,
        'accuracy':  accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, average='weighted', zero_division=0),
        'recall':    recall_score(y_test, y_pred, average='weighted', zero_division=0),
        'f1':        f1_score(y_test, y_pred, average='weighted', zero_division=0),
        'cv_mean':   cv_scores.mean(),
        'cv_std':    cv_scores.std()
    }
    
    print(f'   Accuracy  : {results[name]["accuracy"]:.3f}')
    print(f'   F1 Score  : {results[name]["f1"]:.3f}')
    print(f'   CV F1     : {results[name]["cv_mean"]:.3f} ± {results[name]["cv_std"]:.3f}')

print('\n✅ Tous les modèles entraînés !')

## 6. 📊 Évaluation des performances

In [ ]:
# ── Tableau comparatif ────────────────────────────────────────────────────
df_results = pd.DataFrame({
    name: {
        'Accuracy':  round(v['accuracy'], 4),
        'Précision': round(v['precision'], 4),
        'Rappel':    round(v['recall'], 4),
        'F1 Score':  round(v['f1'], 4),
        'CV F1 (moy)': round(v['cv_mean'], 4),
        'CV F1 (±std)': round(v['cv_std'], 4),
    }
    for name, v in results.items()
}).T

# Mettre en évidence le meilleur
best_model_name = df_results['F1 Score'].astype(float).idxmax()

print('=== Tableau comparatif des modèles ===')
print(df_results.to_string())
print(f'\n🏆 Meilleur modèle : {best_model_name} (F1 = {df_results.loc[best_model_name, "F1 Score"]})')

In [ ]:
# ── Graphique de comparaison des métriques ────────────────────────────────
metrics = ['Accuracy', 'Précision', 'Rappel', 'F1 Score']
x = np.arange(len(models))
width = 0.2

fig, ax = plt.subplots(figsize=(14, 7))
colors = ['#3498db', '#2ecc71', '#e67e22', '#e74c3c']

for i, metric in enumerate(metrics):
    vals = df_results[metric].astype(float).values
    bars = ax.bar(x + i * width, vals, width, label=metric, color=colors[i], alpha=0.85)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.005,
                f'{val:.2f}', ha='center', va='bottom', fontsize=9)

ax.set_xlabel('Modèle')
ax.set_ylabel('Score')
ax.set_title('🏆 Comparaison des modèles de classification', fontsize=14, fontweight='bold')
ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(list(models.keys()), rotation=10)
ax.set_ylim(0, 1.12)
ax.legend(loc='upper right')
ax.axhline(0.8, color='gray', linestyle='--', alpha=0.5, label='Seuil 80%')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('models_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Matrices de confusion ─────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('🔲 Matrices de confusion par modèle', fontsize=14, fontweight='bold')

for ax, (name, res) in zip(axes.flat, results.items()):
    cm = confusion_matrix(y_test, res['y_pred'])
    disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                                   display_labels=le_target.classes_)
    disp.plot(ax=ax, cmap='Blues', colorbar=False)
    ax.set_title(f'{name}\n(Accuracy={res["accuracy"]:.2f}, F1={res["f1"]:.2f})')
    ax.set_xlabel('Prédiction')
    ax.set_ylabel('Réalité')

plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Rapport détaillé du meilleur modèle ──────────────────────────────────
best_pipeline = results[best_model_name]['pipeline']
best_y_pred   = results[best_model_name]['y_pred']

print(f'=== Rapport détaillé : {best_model_name} ===')
print(classification_report(
    y_test, best_y_pred,
    target_names=le_target.classes_
))

## 7. 🔧 Optimisation du meilleur modèle (GridSearch)

In [ ]:
# GridSearch sur Random Forest
print('⏳ Optimisation par GridSearch (RandomForest)...')

param_grid = {
    'model__n_estimators': [50, 100, 200],
    'model__max_depth':    [None, 5, 10],
    'model__min_samples_split': [2, 5]
}

rf_pipeline = Pipeline([
    ('model', RandomForestClassifier(random_state=42))
])

grid_search = GridSearchCV(
    rf_pipeline, param_grid,
    cv=5, scoring='f1_weighted', n_jobs=-1, verbose=0
)
grid_search.fit(X_train, y_train)

print(f'\n✅ Meilleurs paramètres : {grid_search.best_params_}')
print(f'   Meilleur F1 CV      : {grid_search.best_score_:.4f}')

# Score final sur le test
y_pred_opt = grid_search.best_estimator_.predict(X_test)
print(f'   F1 sur test (optimisé) : {f1_score(y_test, y_pred_opt, average="weighted"):.4f}')
print(f'   Accuracy (optimisé)    : {accuracy_score(y_test, y_pred_opt):.4f}')

In [ ]:
# ── Importance des features ───────────────────────────────────────────────
best_rf = grid_search.best_estimator_.named_steps['model']
feature_names = ['Confiance PlantNet', 'Famille botanique', 'Médicinal', 'Toxique', 'Invasif', 'Confiance (relative)']
importances = best_rf.feature_importances_
indices = np.argsort(importances)[::-1]

plt.figure(figsize=(10, 6))
colors_imp = ['#e74c3c' if i == 0 else '#3498db' for i in range(len(importances))]
bars = plt.bar(range(len(importances)),
               importances[indices],
               color=[colors_imp[j] for j in indices],
               alpha=0.85, edgecolor='white')
plt.xticks(range(len(importances)),
           [feature_names[i] for i in indices],
           rotation=20, ha='right')
plt.ylabel('Importance (Gini)')
plt.title('🎯 Importance des features — Random Forest optimisé', fontweight='bold')
for bar, val in zip(bars, importances[indices]):
    plt.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.003,
             f'{val:.3f}', ha='center', va='bottom', fontsize=10)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. 🚀 Exploitation sur de nouvelles données

In [ ]:
# ── Prédiction sur de nouvelles plantes ──────────────────────────────────
final_model = grid_search.best_estimator_

def predict_edibility(scientific_name: str, family: str, confidence: float,
                      is_medicinal: str = 'Non', is_toxic: str = 'Non',
                      is_invasive: str = 'Non') -> dict:
    """Prédit la comestibilité d'une plante à partir de ses caractéristiques."""
    # Encoder les features
    try:
        fam_enc = le_family.transform([family])[0]
    except ValueError:
        fam_enc = 0  # famille inconnue → valeur par défaut
    
    med_enc = le_medicinal.transform([is_medicinal])[0]
    tox_enc = le_toxic.transform([is_toxic])[0]
    inv_enc = le_invasive.transform([is_invasive])[0]
    conf_vs_fam = confidence - df_clean.groupby('family')['confidence'].mean().get(family, df_clean['confidence'].mean())
    
    X_new = pd.DataFrame([{
        'confidence': confidence,
        'family_enc': fam_enc,
        'medicinal_enc': med_enc,
        'toxic_enc': tox_enc,
        'invasive_enc': inv_enc,
        'confidence_vs_family': conf_vs_fam
    }])
    
    pred_enc = final_model.predict(X_new)[0]
    proba    = final_model.predict_proba(X_new)[0]
    pred_label = le_target.inverse_transform([pred_enc])[0]
    
    return {
        'plante': scientific_name,
        'prédiction': pred_label,
        'probabilités': dict(zip(le_target.classes_, proba.round(3)))
    }

# Tests sur de nouvelles plantes
new_plants = [
    ('Solanum lycopersicum', 'Solanaceae', 94.5, 'Non', 'Non', 'Non'),   # Tomate
    ('Nerium oleander',      'Apocynaceae', 87.2, 'Non', 'Oui', 'Non'),  # Laurier rose (toxique)
    ('Musa paradisiaca',     'Musaceae',   91.0, 'Non', 'Non', 'Non'),   # Bananier
    ('Euphorbia tirucalli',  'Euphorbiaceae', 72.0, 'Non', 'Oui', 'Non'), # Plante toxique
]

print('=== Prédictions sur nouvelles plantes ===')
for p in new_plants:
    result = predict_edibility(*p)
    print(f'\n🌿 {result["plante"]}')
    print(f'   Prédiction : {result["prédiction"]}')
    print(f'   Probabilités : {result["probabilités"]}')

## 9. 💾 Sauvegarde du modèle

In [ ]:
import pickle
import json

# Sauvegarder le modèle final
model_artifact = {
    'pipeline': final_model,
    'label_encoders': {
        'family':    le_family,
        'medicinal': le_medicinal,
        'toxic':     le_toxic,
        'invasive':  le_invasive,
        'target':    le_target
    },
    'features': FEATURES,
    'classes':  list(le_target.classes_)
}

with open('whataPlant_model.pkl', 'wb') as f:
    pickle.dump(model_artifact, f)

# Sauvegarder les métriques
metrics_summary = {
    'best_model': best_model_name,
    'accuracy': round(float(df_results.loc[best_model_name, 'Accuracy']), 4),
    'precision': round(float(df_results.loc[best_model_name, 'Précision']), 4),
    'recall': round(float(df_results.loc[best_model_name, 'Rappel']), 4),
    'f1_score': round(float(df_results.loc[best_model_name, 'F1 Score']), 4),
    'optimized_f1': round(float(f1_score(y_test, y_pred_opt, average='weighted')), 4),
    'optimized_accuracy': round(float(accuracy_score(y_test, y_pred_opt)), 4),
    'best_params': str(grid_search.best_params_),
    'dataset_size': len(df_clean),
    'train_size': len(X_train),
    'test_size': len(X_test)
}

with open('model_metrics.json', 'w', encoding='utf-8') as f:
    json.dump(metrics_summary, f, indent=2, ensure_ascii=False)

print('✅ Modèle sauvegardé : whataPlant_model.pkl')
print('✅ Métriques sauvegardées : model_metrics.json')
print(f'\n📋 Résumé final :')
for k, v in metrics_summary.items():
    if k not in ['best_params', 'dataset_size', 'train_size', 'test_size']:
        print(f'   {k}: {v}')
print(f'   Dataset: {metrics_summary["train_size"]} train / {metrics_summary["test_size"]} test')

## 10. 📝 Conclusions

### Ce que nous avons accompli

| Étape | Résultat |
|-------|----------|
| **Problème ML défini** | Classification multi-classes : prédire la comestibilité d'une plante |
| **Données collectées** | Scans réels WhatAPlant (MySQL) + dataset synthétique botanique |
| **Nettoyage** | Valeurs manquantes traitées, labels normalisés, doublons supprimés |
| **Features** | Confiance PlantNet, famille botanique, propriétés (toxique, médicinal, invasif) |
| **Modèles comparés** | Logistic Regression, Random Forest, Gradient Boosting, SVM |
| **Métriques utilisées** | Accuracy, Précision, Rappel, F1-Score, Cross-Validation 5-fold |
| **Optimisation** | GridSearchCV sur Random Forest (n_estimators, max_depth, min_samples) |
| **Exploitation** | Fonction `predict_edibility()` prête pour intégration dans l'application |

### Enseignements

- La **famille botanique** est le facteur le plus discriminant pour prédire la comestibilité
- Le score de **toxicité** est fortement corrélé négativement avec la comestibilité
- Le **Random Forest optimisé** offre le meilleur équilibre précision/robustesse
- L'intégration dans WhatAPlant permet d'**enrichir les prédictions PlantNet** avec une couche ML locale